In [8]:
import json
import pandas as pd

In [9]:
PATH_INPUT = '../data/input/'
PATH_OUTPUT = '../data/output/'

In [10]:
def carregar_taxonomia(caminho_arquivo):
    """
    Abre o arquivo JSON de regras e o converte em um dicionário Python.
    """
    try:
        with open(caminho_arquivo, 'r', encoding='utf-8') as arquivo:
            dicionario_taxonomia = json.load(arquivo)
        return dicionario_taxonomia
        
    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
        return None
    except json.JSONDecodeError:
        print(f"Erro: O arquivo '{caminho_arquivo}' não é um JSON válido.")
        return None

In [11]:
def gerar_prompt_ancorado(nome_classe, regras, ancoras_academicas, sub_escopos=None,numero=100):
    """
    Gera o prompt em inglês fundindo as âncoras do artigo com a geração em lote.
    """
    ancoras_formatadas = ", ".join([f'"{ancora}"' for ancora in ancoras_academicas])
    qtd_ancoras = len(ancoras_academicas)
    
    if sub_escopos:
        escopos_texto = "\n".join([f"{i+1}. {escopo}" for i, escopo in enumerate(sub_escopos)])
        bloco_expansao = f"""Expansion Scopes:
To ensure high vocabulary diversity for the remaining sentences, split them equally across these sub-scopes:
{escopos_texto}"""
    else:
        bloco_expansao = f"""Expansion Scopes:
To ensure high vocabulary diversity, explore all possible peripheral concepts related to: {regras}"""

    prompt = f"""Act as a data engineer. Generate a valid JSON object containing exactly ONE key: "{nome_classe}". 
The value must be a list of exactly {numero} distinct, short sentences in English describing data dictionary variables.

Academic Anchors (MANDATORY):
The first {qtd_ancoras} sentences generated MUST be explicit descriptions of these exact concepts from scientific literature:
[{ancoras_formatadas}]

{bloco_expansao}

1. Noun Phrase: Start directly with the core business noun.
2. Passive Voice: Structure the sentence around the measured object being acted upon.
3. Direct Action: Start with an infinitive verb (To + verb) or an action-derived noun.

Absolute Writing Constraints

- Zero-Gerund Policy: It is STRICTLY FORBIDDEN to use words ending in "-ing" acting as verbs, participles, or connectors (e.g., "using", "indicating", "measuring"). Exception granted ONLY for canonical domain nouns (e.g., "housing", "banking", "schooling").
- Banned Prefixes: Do not start sentences with "Variable that", "This variable", "Refers to", "Indicates", or "Measures".
- Length: Keep sentences strictly between 8 and 25 words.

Output format:
{{"{nome_classe}": ["sentence_1", "sentence_2", ..., "sentence_{numero}"]}}
"""
    return prompt

In [12]:
def gerar_arquivos(df_ofc_vars: pd.DataFrame, list_macro : list[str], list_macro_txt : list[str]) -> None:
    for macro, nome_arquivo in zip(list_macro,list_macro_txt):
        dict_taxonomia = carregar_taxonomia(PATH_INPUT + 'MacroRules.json')

        variaveis_artigo = df_ofc_vars[df_ofc_vars['macro'] == macro]['vars'].to_list()

        prompt_pronto = gerar_prompt_ancorado(
            nome_classe=macro, 
            regras="", 
            ancoras_academicas=variaveis_artigo, 
            sub_escopos=dict_taxonomia[macro],
            numero=100
        )

        with open(PATH_OUTPUT + 'prompt/' + nome_arquivo + '.txt', "w", encoding="utf-8") as arquivo:
            arquivo.write(prompt_pronto)

In [13]:
df_ofc_vars = pd.read_csv(PATH_INPUT + 'PosArticleVars.csv')
list_macro = df_ofc_vars['macro'].unique().tolist()
list_macro_txt = [i[:6] for i in list_macro]

In [14]:
if __name__ == "__main__":
    gerar_arquivos(df_ofc_vars, list_macro, list_macro_txt)